<a href="https://colab.research.google.com/github/bahmedx/730/blob/main/W5P2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# CELL 1: LIBRARIES AND PROJECT CONFIGURATION
# ============================================================

import pandas as pd
import numpy as np
import requests

import matplotlib.pyplot as plt
import seaborn as sns

import statsmodels.formula.api as smf

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from getpass import getpass
import warnings

warnings.filterwarnings("ignore")

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Analysis period
START_YEAR = 2015
END_YEAR = 2025

# Chronological train/test split
TRAIN_END_YEAR = 2023
TEST_START_YEAR = 2024

# Counterfactual policy scenarios
POLICY_SCENARIOS = {
    "Baseline": 0.00,
    "5% Reduction": 0.05,
    "10% Reduction": 0.10,
    "15% Reduction": 0.15
}

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

print("Project configuration complete.")
print(f"Historical period: {START_YEAR}-{END_YEAR}")
print(f"Training period: {START_YEAR}-{TRAIN_END_YEAR}")
print(f"Testing period: {TEST_START_YEAR}-{END_YEAR}")

In [ ]:
# ============================================================
# CELL 2: EIA API KEY AND DATA-DOWNLOAD FUNCTION
# ============================================================

EIA_API_KEY = getpass("Enter your EIA API key: ")

if not EIA_API_KEY.strip():
    raise ValueError("A valid EIA API key is required.")


def get_eia_retail_data(api_key, start_year, end_year):
    """
    Download monthly state-level retail electricity data
    from the EIA API using pagination.
    """

    url = "https://api.eia.gov/v2/electricity/retail-sales/data/"

    page_size = 5000
    offset = 0
    all_rows = []

    while True:

        params = [
            ("api_key", api_key),
            ("frequency", "monthly"),
            ("data[0]", "sales"),
            ("data[1]", "revenue"),
            ("data[2]", "price"),
            ("data[3]", "customers"),
            ("start", str(start_year)),
            ("end", str(end_year)),
            ("offset", offset),
            ("length", page_size)
        ]

        response = requests.get(
            url,
            params=params,
            timeout=60
        )

        response.raise_for_status()
        payload = response.json()

        rows = payload.get("response", {}).get("data", [])

        if not rows:
            break

        all_rows.extend(rows)

        print(
            f"Downloaded {len(all_rows):,} records",
            end="\r"
        )

        if len(rows) < page_size:
            break

        offset += page_size

    print(f"\nDownload complete: {len(all_rows):,} records.")

    return pd.DataFrame(all_rows)

In [ ]:
# ============================================================
# CELL 3: DOWNLOAD HISTORICAL EIA ELECTRICITY DATA
# ============================================================

eia_raw = get_eia_retail_data(
    api_key=EIA_API_KEY,
    start_year=START_YEAR,
    end_year=END_YEAR
)

print("\nRaw dataset shape:")
print(eia_raw.shape)

display(eia_raw.head())

In [ ]:
# ============================================================
# CELL 4: INITIAL DATA INSPECTION
# ============================================================

print("DATASET OVERVIEW")
print("=" * 60)

print(f"Rows: {eia_raw.shape,}")
print(f"Columns: {eia_raw.shape[1]}")

print("\nColumn names:")
print(eia_raw.columns.tolist())

print("\nFirst five observations:")
display(eia_raw.head())

print("\nSector coverage:")
display(
    eia_raw[["sectorid", "sectorName"]]
    .drop_duplicates()
    .sort_values("sectorid")
)

print("\nMissing values:")
missing_raw = pd.DataFrame({
    "Missing": eia_raw.isna().sum(),
    "Percent": (eia_raw.isna().mean() * 100).round(2)
})

display(missing_raw)

In [ ]:
# ============================================================
# CELL 5: CLEAN AND RESHAPE DATA
# ============================================================

df = eia_raw.copy()

# Convert date
df["period"] = pd.to_datetime(
    df["period"],
    errors="coerce"
)

# Convert EIA numerical fields
numeric_cols = ["sales", "revenue", "price", "customers"]

for col in numeric_cols:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

# Standardize text fields
df["sector"] = (
    df["sectorName"]
    .astype(str)
    .str.strip()
    .str.lower()
)

df["state"] = (
    df["stateDescription"]
    .astype(str)
    .str.strip()
)

# Keep residential, commercial, and industrial sectors
keep_sectors = [
    "residential",
    "commercial",
    "industrial"
]

df = df[
    df["sector"].isin(keep_sectors)
].copy()

# Exclude national totals
df = df[
    ~df["state"].isin(
        ["United States", "U.S. Total"]
    )
].copy()

# Create time variables
df["year"] = df["period"].dt.year
df["month"] = df["period"].dt.month

# Reshape to one state-month observation
state_month = df.pivot_table(
    index=["period", "year", "month", "state"],
    columns="sector",
    values=["sales", "price"],
    aggfunc="first"
)

state_month.columns = [
    f"{sector}_{measure}"
    for measure, sector in state_month.columns
]

state_month = (
    state_month
    .reset_index()
    .sort_values(["state", "period"])
    .reset_index(drop=True)
)

print("Clean state-month dataset:")
print(state_month.shape)

display(state_month.head())

In [ ]:
# ============================================================
# CELL 6: MISSING-DATA ANALYSIS AND PREPROCESSING
# ============================================================

required_variables = [
    "residential_price",
    "residential_sales",
    "commercial_sales",
    "industrial_sales"
]

missing_summary = pd.DataFrame({
    "Missing": state_month[required_variables].isna().sum(),
    "Percent": (
        state_month[required_variables]
        .isna()
        .mean()
        .mul(100)
        .round(2)
    )
})

print("Missing-value summary:")
display(missing_summary)

# Remove observations missing variables required for analysis.
# Mean imputation is avoided because it could distort
# state-level electricity demand relationships.

analysis_df = state_month.dropna(
    subset=required_variables
).copy()

# Remove impossible non-positive observations
analysis_df = analysis_df[
    (analysis_df["residential_price"] > 0) &
    (analysis_df["residential_sales"] > 0) &
    (analysis_df["commercial_sales"] > 0) &
    (analysis_df["industrial_sales"] > 0)
].copy()

print(f"Observations retained: {len(analysis_df):,}")
print(f"States/regions: {analysis_df['state'].nunique()}")
print(
    f"Period: {analysis_df['period'].min().date()} "
    f"to {analysis_df['period'].max().date()}"
)

In [ ]:
# ============================================================
# CELL 7: FEATURE ENGINEERING
# ============================================================

analysis_df = (
    analysis_df
    .sort_values(["state", "period"])
    .copy()
)

# Log electricity demand
analysis_df["log_commercial_sales"] = np.log1p(
    analysis_df["commercial_sales"]
)

analysis_df["log_residential_sales"] = np.log1p(
    analysis_df["residential_sales"]
)

analysis_df["log_industrial_sales"] = np.log1p(
    analysis_df["industrial_sales"]
)

# Year-over-year commercial electricity-demand growth
analysis_df["commercial_growth"] = (
    analysis_df
    .groupby("state")["commercial_sales"]
    .pct_change(12)
)

# Lagged residential electricity price
analysis_df["price_lag12"] = (
    analysis_df
    .groupby("state")["residential_price"]
    .shift(12)
)

# Seasonal controls
analysis_df["month_sin"] = np.sin(
    2 * np.pi * analysis_df["month"] / 12
)

analysis_df["month_cos"] = np.cos(
    2 * np.pi * analysis_df["month"] / 12
)

model_variables = [
    "residential_price",
    "log_commercial_sales",
    "log_residential_sales",
    "log_industrial_sales",
    "commercial_growth",
    "price_lag12",
    "month_sin",
    "month_cos"
]

analysis_df = analysis_df.dropna(
    subset=model_variables
).copy()

print(f"Final modeling observations: {len(analysis_df):,}")

display(
    analysis_df[model_variables]
    .describe()
    .T
    .round(3)
)

In [ ]:
# ============================================================
# CELL 8: EXPLORATORY DATA ANALYSIS
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Residential electricity price over time
monthly_price = (
    analysis_df
    .groupby("period")["residential_price"]
    .mean()
)

axes[0].plot(
    monthly_price.index,
    monthly_price.values
)

axes[0].set_title(
    "Average Residential Electricity Price Over Time"
)
axes[0].set_xlabel("Year")
axes[0].set_ylabel("Price (cents/kWh)")


# Commercial electricity sales over time
monthly_commercial = (
    analysis_df
    .groupby("period")["commercial_sales"]
    .mean()
)

axes[1].plot(
    monthly_commercial.index,
    monthly_commercial.values
)

axes[1].set_title(
    "Average Commercial Electricity Sales Over Time"
)
axes[1].set_xlabel("Year")
axes[1].set_ylabel("Commercial Electricity Sales")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 9: TIME-BASED TRAIN/TEST SPLIT
# ============================================================

train_df = analysis_df[
    analysis_df["year"] <= TRAIN_END_YEAR
].copy()

test_df = analysis_df[
    analysis_df["year"] >= TEST_START_YEAR
].copy()

if train_df.empty or test_df.empty:
    raise ValueError(
        "Training or testing dataset is empty. "
        "Check the available years."
    )

print("TRAINING DATA")
print(
    train_df["period"].min().date(),
    "to",
    train_df["period"].max().date()
)
print(f"Observations: {len(train_df):,}")

print("\nTESTING DATA")
print(
    test_df["period"].min().date(),
    "to",
    test_df["period"].max().date()
)
print(f"Observations: {len(test_df):,}")

In [ ]:
# ============================================================
# CELL 10: NORMALIZE MODEL PREDICTORS
# ============================================================

continuous_features = [
    "log_commercial_sales",
    "log_residential_sales",
    "log_industrial_sales",
    "commercial_growth",
    "price_lag12"
]

scaler = StandardScaler()

train_scaled = train_df.copy()
test_scaled = test_df.copy()

# Fit scaler only on training observations
train_scaled[continuous_features] = (
    scaler.fit_transform(
        train_df[continuous_features]
    )
)

# Apply same transformation to test observations
test_scaled[continuous_features] = (
    scaler.transform(
        test_df[continuous_features]
    )
)

print("Predictors normalized successfully.")

print("\nStandardized training means:")
display(
    train_scaled[
        continuous_features
    ].mean().round(3).to_frame("Mean")
)

In [ ]:
# ============================================================
# CELL 11: MODEL IMPLEMENTATION
# ============================================================

formula = """
residential_price
~ log_commercial_sales
+ log_residential_sales
+ log_industrial_sales
+ commercial_growth
+ price_lag12
+ month_sin
+ month_cos
+ C(state)
+ C(year)
"""

price_model = smf.ols(
    formula=formula,
    data=train_scaled
).fit(
    cov_type="cluster",
    cov_kwds={
        "groups": train_scaled["state"]
    }
)

print("Model fitted successfully.")
print(f"Training observations: {int(price_model.nobs):,}")
print(f"Adjusted R²: {price_model.rsquared_adj:.3f}")

In [ ]:
# ============================================================
# CELL 12: MODEL EVALUATION
# ============================================================

train_predictions = price_model.predict(
    train_scaled
)

test_predictions = price_model.predict(
    test_scaled
)

train_r2 = r2_score(
    train_scaled["residential_price"],
    train_predictions
)

test_r2 = r2_score(
    test_scaled["residential_price"],
    test_predictions
)

test_mae = mean_absolute_error(
    test_scaled["residential_price"],
    test_predictions
)

test_rmse = np.sqrt(
    mean_squared_error(
        test_scaled["residential_price"],
        test_predictions
    )
)

print("MODEL PERFORMANCE")
print("=" * 45)
print(f"Training R²: {train_r2:.3f}")
print(f"Testing R²:  {test_r2:.3f}")
print(f"Testing MAE: {test_mae:.3f} cents/kWh")
print(f"Testing RMSE: {test_rmse:.3f} cents/kWh")


# Actual vs. predicted plot
plt.figure(figsize=(7, 6))

plt.scatter(
    test_scaled["residential_price"],
    test_predictions,
    alpha=0.45
)

low = min(
    test_scaled["residential_price"].min(),
    test_predictions.min()
)

high = max(
    test_scaled["residential_price"].max(),
    test_predictions.max()
)

plt.plot(
    [low, high],
    [low, high],
    "r--"
)

plt.title(
    "Actual vs. Predicted Residential Electricity Prices"
)
plt.xlabel("Actual Price (cents/kWh)")
plt.ylabel("Predicted Price (cents/kWh)")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 13: MAIN MODEL COEFFICIENTS
# ============================================================

main_variables = [
    "log_commercial_sales",
    "log_residential_sales",
    "log_industrial_sales",
    "commercial_growth",
    "price_lag12",
    "month_sin",
    "month_cos"
]

coef_table = pd.DataFrame({
    "Coefficient": price_model.params[main_variables],
    "Std. Error": price_model.bse[main_variables],
    "P-value": price_model.pvalues[main_variables]
})

coef_table["Significant at 5%"] = (
    coef_table["P-value"] < 0.05
)

display(
    coef_table.round(4)
)

In [ ]:
# ============================================================
# CELL 14: COUNTERFACTUAL POLICY SIMULATION
# ============================================================

def prepare_for_prediction(df_original):
    """
    Apply the training-data scaler to a new dataset
    before generating model predictions.
    """

    df_scaled = df_original.copy()

    df_scaled[continuous_features] = scaler.transform(
        df_scaled[continuous_features]
    )

    return df_scaled


# Baseline predictions
baseline_data = prepare_for_prediction(test_df)

baseline_predictions = price_model.predict(
    baseline_data
)

results = []

for scenario, reduction in POLICY_SCENARIOS.items():

    cf = test_df.copy()

    # Apply hypothetical commercial demand reduction
    cf["commercial_sales"] = (
        cf["commercial_sales"] *
        (1 - reduction)
    )

    # Recalculate affected model feature
    cf["log_commercial_sales"] = np.log1p(
        cf["commercial_sales"]
    )

    # Prepare model-ready counterfactual data
    cf_scaled = prepare_for_prediction(cf)

    # Generate counterfactual predictions
    cf_predictions = price_model.predict(
        cf_scaled
    )

    results.append({
        "Scenario": scenario,
        "Demand Reduction (%)": reduction * 100,
        "Predicted Price (cents/kWh)":
            cf_predictions.mean(),
        "Price Change (cents/kWh)":
            (
                cf_predictions -
                baseline_predictions
            ).mean()
    })


scenario_results = pd.DataFrame(results)

scenario_results["Price Change (%)"] = (
    scenario_results["Price Change (cents/kWh)"]
    /
    baseline_predictions.mean()
    * 100
)

display(
    scenario_results.round(4)
)

In [ ]:
# ============================================================
# CELL 15: COUNTERFACTUAL RESULTS VISUALIZATION
# ============================================================

plt.figure(figsize=(8, 5))

sns.lineplot(
    data=scenario_results,
    x="Demand Reduction (%)",
    y="Predicted Price (cents/kWh)",
    marker="o",
    linewidth=2
)

plt.title(
    "Predicted Residential Electricity Price\n"
    "Under Alternative Demand-Management Scenarios"
)

plt.xlabel(
    "Commercial Electricity Demand Reduction (%)"
)

plt.ylabel(
    "Predicted Residential Price (cents/kWh)"
)

plt.xticks(
    scenario_results["Demand Reduction (%)"]
)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 16: FINAL RESULTS SUMMARY
# ============================================================

baseline = scenario_results.loc[
    scenario_results["Scenario"] == "Baseline"
].iloc[0]

policy_10 = scenario_results.loc[
    scenario_results["Scenario"] == "10% Reduction"
].iloc[0]


print("COUNTERFACTUAL ANALYSIS SUMMARY")
print("=" * 60)

print("\nMODEL PERFORMANCE")
print(f"Test R²:   {test_r2:.3f}")
print(f"Test MAE:  {test_mae:.3f} cents/kWh")
print(f"Test RMSE: {test_rmse:.3f} cents/kWh")

print("\nBASELINE")
print(
    "Predicted residential electricity price: "
    f"{baseline['Predicted Price (cents/kWh)']:.3f} "
    "cents/kWh"
)

print("\n10% DEMAND-REDUCTION POLICY")
print(
    "Predicted residential electricity price: "
    f"{policy_10['Predicted Price (cents/kWh)']:.3f} "
    "cents/kWh"
)

print(
    "Estimated price change: "
    f"{policy_10['Price Change (cents/kWh)']:.3f} "
    "cents/kWh"
)

print(
    "Estimated percentage change: "
    f"{policy_10['Price Change (%)']:.2f}%"
)

print("\nINTERPRETATION")

direction = (
    "decrease"
    if policy_10["Price Change (cents/kWh)"] < 0
    else "increase"
)

print(
    "Under the fitted model, a hypothetical 10% reduction "
    "in commercial electricity demand is associated with a "
    f"{direction} in predicted residential electricity prices."
)

print(
    "\nCAUTION: This result is a model-based counterfactual "
    "estimate and should not be interpreted as definitive "
    "evidence that AI or data centers causally determine "
    "residential electricity prices."
)

In [ ]:
# ============================================================
# CELL 17: ENGINEER ELECTRICITY-MARKET CONTROL VARIABLES
# ============================================================

model_df = (
    model_df
    .sort_values(["state", "period"])
    .copy()
)

# Residential sales growth
model_df["residential_sales_growth"] = (
    model_df
    .groupby("state")["residential_sales"]
    .pct_change(12)
)

# Industrial sales growth
model_df["industrial_sales_growth"] = (
    model_df
    .groupby("state")["industrial_sales"]
    .pct_change(12)
)

# Lagged residential price
model_df["residential_price_lag12"] = (
    model_df
    .groupby("state")["residential_price"]
    .shift(12)
)

# Log transformations for skewed sales measures
model_df["log_commercial_sales"] = np.log1p(
    model_df["commercial_sales"]
)

model_df["log_residential_sales"] = np.log1p(
    model_df["residential_sales"]
)

model_df["log_industrial_sales"] = np.log1p(
    model_df["industrial_sales"]
)

# Seasonal variables
model_df["month_sin"] = np.sin(
    2 * np.pi * model_df["month"] / 12
)

model_df["month_cos"] = np.cos(
    2 * np.pi * model_df["month"] / 12
)

print("Control variables created successfully.")

In [ ]:
# ============================================================
# CELL 18: PREPARE FINAL MODELING DATASET
# ============================================================

model_features = [
    "residential_price",
    "dc_load_growth_exposure",
    "commercial_sales_growth",
    "residential_sales_growth",
    "industrial_sales_growth",
    "log_commercial_sales",
    "log_residential_sales",
    "log_industrial_sales",
    "residential_price_lag12",
    "month_sin",
    "month_cos"
]

print("Missing values before filtering:")

missing_model = pd.DataFrame({
    "Missing": model_df[model_features].isna().sum(),
    "Percent": (
        model_df[model_features]
        .isna()
        .mean()
        .mul(100)
        .round(2)
    )
})

display(missing_model)

# Remove only records unusable for the model.
# We do not mean-impute time-series economic observations.
analysis_df = model_df.dropna(
    subset=model_features
).copy()

# Remove impossible/non-positive price observations
analysis_df = analysis_df[
    analysis_df["residential_price"] > 0
].copy()

print(
    f"\nFinal modeling observations: "
    f"{len(analysis_df):,}"
)

print(
    f"States represented: "
    f"{analysis_df['state'].nunique()}"
)

print(
    f"Period: "
    f"{analysis_df['period'].min().date()} to "
    f"{analysis_df['period'].max().date()}"
)

In [ ]:
# ============================================================
# CELL 19: DATA-CENTER EXPOSURE EDA
# ============================================================

state_exposure_plot = (
    dc_exposure
    .sort_values(
        "data_center_count",
        ascending=False
    )
    .head(15)
)

plt.figure(figsize=(10, 6))

sns.barplot(
    data=state_exposure_plot,
    x="data_center_count",
    y="state",
    color="steelblue"
)

plt.title(
    "States with the Highest Data-Center Facility Exposure"
)
plt.xlabel("Number of Listed Data Centers")
plt.ylabel("State")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 20: TIME-BASED TRAIN/TEST SPLIT
# ============================================================

train_df = analysis_df[
    analysis_df["year"] <= TRAIN_END_YEAR
].copy()

test_df = analysis_df[
    analysis_df["year"] >= TEST_START_YEAR
].copy()

# Validation
assert train_df["period"].max() < test_df["period"].min(), (
    "Training and testing periods overlap."
)

print("TIME-BASED DATA SPLIT")
print("=" * 50)

print(
    f"Training observations: "
    f"{len(train_df):,}"
)

print(
    f"Training period: "
    f"{train_df['period'].min().date()} to "
    f"{train_df['period'].max().date()}"
)

print()

print(
    f"Testing observations: "
    f"{len(test_df):,}"
)

print(
    f"Testing period: "
    f"{test_df['period'].min().date()} to "
    f"{test_df['period'].max().date()}"
)

print()

print(
    f"Training states: "
    f"{train_df['state'].nunique()}"
)

print(
    f"Testing states: "
    f"{test_df['state'].nunique()}"
)

In [ ]:
# ============================================================
# CELL 21: NORMALIZE CONTINUOUS PREDICTORS
# ============================================================

continuous_features = [
    "dc_load_growth_exposure",
    "commercial_sales_growth",
    "residential_sales_growth",
    "industrial_sales_growth",
    "log_commercial_sales",
    "log_residential_sales",
    "log_industrial_sales",
    "residential_price_lag12"
]

scaler = StandardScaler()

# Fit ONLY on training data
train_df.loc[:, continuous_features] = (
    scaler.fit_transform(
        train_df[continuous_features]
    )
)

# Apply training transformation to test data
test_df.loc[:, continuous_features] = (
    scaler.transform(
        test_df[continuous_features]
    )
)

print("Continuous variables normalized successfully.")

print(
    "\nApproximate standardized training means:"
)

display(
    train_df[
        continuous_features
    ].mean().round(3).to_frame("Mean")
)

In [ ]:
# ============================================================
# CELL 22: INITIAL ELECTRICITY PRICE MODEL
# ============================================================

model_formula = """
residential_price
~ dc_load_growth_exposure
+ commercial_sales_growth
+ residential_sales_growth
+ industrial_sales_growth
+ log_commercial_sales
+ log_residential_sales
+ log_industrial_sales
+ residential_price_lag12
+ month_sin
+ month_cos
+ C(state)
+ C(year)
"""

price_model = smf.ols(
    formula=model_formula,
    data=train_df
).fit(
    cov_type="cluster",
    cov_kwds={
        "groups": train_df["state"]
    }
)

print("Model fitted successfully.")

print(
    price_model.summary()
)

In [ ]:
# ============================================================
# CELL 23: MODEL PERFORMANCE
# ============================================================

# Predictions
train_pred = price_model.predict(train_df)
test_pred = price_model.predict(test_df)

# Metrics
train_r2 = r2_score(
    train_df["residential_price"],
    train_pred
)

test_r2 = r2_score(
    test_df["residential_price"],
    test_pred
)

test_mae = mean_absolute_error(
    test_df["residential_price"],
    test_pred
)

test_rmse = np.sqrt(
    mean_squared_error(
        test_df["residential_price"],
        test_pred
    )
)

print("MODEL PERFORMANCE")
print("=" * 45)

print(f"Training R²: {train_r2:.3f}")
print(f"Testing R²:  {test_r2:.3f}")
print(f"Testing MAE: {test_mae:.3f} cents/kWh")
print(f"Testing RMSE: {test_rmse:.3f} cents/kWh")

In [ ]:
# ============================================================
# CELL 24: ACTUAL VS. PREDICTED PRICES
# ============================================================

plot_df = test_df.copy()

plot_df["predicted_price"] = test_pred

plt.figure(figsize=(7, 6))

sns.scatterplot(
    data=plot_df,
    x="residential_price",
    y="predicted_price",
    alpha=0.5
)

min_value = min(
    plot_df["residential_price"].min(),
    plot_df["predicted_price"].min()
)

max_value = max(
    plot_df["residential_price"].max(),
    plot_df["predicted_price"].max()
)

plt.plot(
    [min_value, max_value],
    [min_value, max_value],
    linestyle="--",
    color="red"
)

plt.title(
    "Actual vs. Predicted Residential Electricity Prices"
)

plt.xlabel(
    "Actual Price (cents/kWh)"
)

plt.ylabel(
    "Predicted Price (cents/kWh)"
)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 25: COUNTERFACTUAL SCENARIO GENERATOR
# ============================================================

# Start from unscaled test observations
counterfactual_base = analysis_df[
    analysis_df["year"] >= TEST_START_YEAR
].copy()

def create_counterfactual(
    base_df,
    demand_reduction,
    scaler,
    continuous_features
):
    """
    Create a counterfactual electricity-demand scenario.

    Parameters
    ----------
    base_df : DataFrame
        Unscaled test-period observations.

    demand_reduction : float
        Proportional reduction in the modeled
        data-center/load-growth exposure.

        Example:
        0.10 = 10% reduction

    scaler : fitted StandardScaler
        Scaler fitted using training observations.

    continuous_features : list
        Variables standardized for the regression.

    Returns
    -------
    DataFrame
        Model-ready counterfactual observations.
    """

    cf = base_df.copy()

    # Reduce modeled exposure
    cf["dc_load_growth_exposure"] *= (
        1 - demand_reduction
    )

    # Standardize using TRAINING scaler
    cf.loc[:, continuous_features] = (
        scaler.transform(
            cf[continuous_features]
        )
    )

    return cf

In [ ]:
# ============================================================
# CELL 26: RUN COUNTERFACTUAL SIMULATIONS
# ============================================================

scenario_results = []

for scenario_name, change in POLICY_SCENARIOS.items():

    reduction = abs(change)

    cf_df = create_counterfactual(
        base_df=counterfactual_base,
        demand_reduction=reduction,
        scaler=scaler,
        continuous_features=continuous_features
    )

    cf_predictions = price_model.predict(cf_df)

    scenario_results.append({
        "Scenario": scenario_name,
        "Demand Reduction": reduction,
        "Predicted Residential Price":
            cf_predictions.mean()
    })

scenario_results = pd.DataFrame(
    scenario_results
)

baseline_price = (
    scenario_results
    .loc[
        scenario_results["Scenario"] == "Baseline",
        "Predicted Residential Price"
    ]
    .iloc[0]
)

scenario_results["Price Change"] = (
    scenario_results["Predicted Residential Price"]
    - baseline_price
)

scenario_results["Percent Price Change"] = (
    scenario_results["Price Change"]
    / baseline_price
    * 100
)

display(
    scenario_results.round(3)
)

In [ ]:
# ============================================================
# CELL 27: COUNTERFACTUAL POLICY IMPACT
# ============================================================

plt.figure(figsize=(8, 5))

sns.lineplot(
    data=scenario_results,
    x="Demand Reduction",
    y="Predicted Residential Price",
    marker="o",
    linewidth=2
)

plt.title(
    "Counterfactual Residential Electricity Prices"
)

plt.xlabel(
    "Modeled Data-Center Demand-Exposure Reduction"
)

plt.ylabel(
    "Predicted Residential Price (cents/kWh)"
)

plt.xticks(
    scenario_results["Demand Reduction"],
    [
        f"{x:.0%}"
        for x in scenario_results["Demand Reduction"]
    ]
)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 28: STATE-LEVEL COUNTERFACTUAL EFFECTS
# ============================================================

# Baseline
baseline_cf = create_counterfactual(
    base_df=counterfactual_base,
    demand_reduction=0.00,
    scaler=scaler,
    continuous_features=continuous_features
)

baseline_cf["baseline_prediction"] = (
    price_model.predict(baseline_cf)
)

# 10% reduction scenario
cf_10 = create_counterfactual(
    base_df=counterfactual_base,
    demand_reduction=0.10,
    scaler=scaler,
    continuous_features=continuous_features
)

baseline_cf["cf_10_prediction"] = (
    price_model.predict(cf_10)
)

baseline_cf["estimated_price_change"] = (
    baseline_cf["cf_10_prediction"]
    - baseline_cf["baseline_prediction"]
)

state_effects = (
    baseline_cf
    .groupby("state", as_index=False)
    .agg(
        data_center_count=(
            "data_center_count",
            "first"
        ),
        baseline_price=(
            "baseline_prediction",
            "mean"
        ),
        counterfactual_price=(
            "cf_10_prediction",
            "mean"
        ),
        estimated_price_change=(
            "estimated_price_change",
            "mean"
        )
    )
)

state_effects = state_effects.sort_values(
    "estimated_price_change"
)

display(
    state_effects.head(15).round(3)
)

In [ ]:
# ============================================================
# CELL 29: STATE-LEVEL COUNTERFACTUAL IMPACT
# ============================================================

plot_states = (
    state_effects
    .assign(
        absolute_effect=lambda x:
            x["estimated_price_change"].abs()
    )
    .nlargest(
        15,
        "absolute_effect"
    )
    .sort_values(
        "estimated_price_change"
    )
)

plt.figure(figsize=(10, 7))

sns.barplot(
    data=plot_states,
    x="estimated_price_change",
    y="state",
    color="steelblue"
)

plt.axvline(
    0,
    color="black",
    linewidth=1
)

plt.title(
    "Estimated Residential Price Change Under a 10% "
    "Demand-Exposure Reduction"
)

plt.xlabel(
    "Estimated Price Change (cents/kWh)"
)

plt.ylabel("State")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 30: SENSITIVITY ANALYSIS
# ============================================================

sensitivity_levels = np.arange(
    0.00,
    0.31,
    0.025
)

sensitivity_results = []

for reduction in sensitivity_levels:

    cf = create_counterfactual(
        base_df=counterfactual_base,
        demand_reduction=reduction,
        scaler=scaler,
        continuous_features=continuous_features
    )

    prediction = price_model.predict(cf)

    sensitivity_results.append({
        "Demand Reduction": reduction,
        "Predicted Price": prediction.mean()
    })

sensitivity_results = pd.DataFrame(
    sensitivity_results
)

plt.figure(figsize=(8, 5))

sns.lineplot(
    data=sensitivity_results,
    x="Demand Reduction",
    y="Predicted Price",
    marker="o"
)

plt.title(
    "Sensitivity of Predicted Electricity Prices "
    "to Demand Management"
)

plt.xlabel(
    "Modeled Demand-Exposure Reduction"
)

plt.ylabel(
    "Predicted Residential Price (cents/kWh)"
)

plt.gca().xaxis.set_major_formatter(
    plt.FuncFormatter(
        lambda x, _: f"{x:.0%}"
    )
)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# CELL 31: FINAL COUNTERFACTUAL SUMMARY
# ============================================================

scenario_10 = scenario_results.loc[
    scenario_results["Scenario"] == "10% Reduction"
].iloc[0]

print("COUNTERFACTUAL ANALYSIS SUMMARY")
print("=" * 55)

print(
    f"Test RMSE: "
    f"{test_rmse:.3f} cents/kWh"
)

print(
    f"Test MAE: "
    f"{test_mae:.3f} cents/kWh"
)

print(
    f"Test R²: "
    f"{test_r2:.3f}"
)

print()

print(
    f"Baseline predicted residential price: "
    f"{baseline_price:.3f} cents/kWh"
)

print(
    f"10% reduction scenario: "
    f"{scenario_10['Predicted Residential Price']:.3f} "
    f"cents/kWh"
)

print(
    f"Estimated change: "
    f"{scenario_10['Price Change']:.3f} cents/kWh"
)

print(
    f"Estimated percentage change: "
    f"{scenario_10['Percent Price Change']:.2f}%"
)

print()
print(
    "Interpretation: These estimates represent "
    "model-based counterfactual associations and "
    "should not be interpreted as definitive causal effects."
)